
T5-Base Architecture Inspection (HuggingFace Transformers) 

In [1]:
import os
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import math
import torch
import pandas as pd

torch.set_num_threads(1)

import transformers.utils.import_utils as iu
iu.is_torchvision_available.cache_clear()

from transformers import T5Config, T5TokenizerFast, T5ForConditionalGeneration

print("PyTorch version:", torch.__version__)


PyTorch version: 2.10.0+cpu


In [6]:
from transformers import T5Config, T5Tokenizer, T5ForConditionalGeneration

model_name = "t5-base"

config = T5Config.from_pretrained(model_name)
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

model.eval()

print("Model loaded successfully")

config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded successfully


Part 1: Configuration inspection

In [7]:

config_table = pd.DataFrame([{
    "d_model": config.d_model,
    "num_layers": config.num_layers,
    "num_heads": config.num_heads,
    "d_ff": config.d_ff,
    "vocab_size": config.vocab_size,
    "feed_forward_proj": config.feed_forward_proj,
}])

head_dim = config.d_model // config.num_heads

config_table

,d_model,num_layers,num_heads,d_ff,vocab_size,feed_forward_proj
0,768,12,12,3072,32128,relu


In [11]:
encoder_block0 = model.encoder.block[0]

sa = encoder_block0.layer[0].SelfAttention
ffn = encoder_block0.layer[1].DenseReluDense

block_table = pd.DataFrame([{
    "sa.n_heads": sa.n_heads,
    "sa.q.in_features": sa.q.in_features,
    "sa.q.out_features": sa.q.out_features,
    "ffn.wi.in_features": ffn.wi.in_features,
    "ffn.wi.out_features": ffn.wi.out_features,
}])

block_table

,sa.n_heads,sa.q.in_features,sa.q.out_features,ffn.wi.in_features,ffn.wi.out_features
0,12,768,768,768,3072


Part 2: Single forward pass on the summarization-style input

In [13]:
text = """summarize: Federal banking regulators are preparing to release a set of proposals to loosen requirements on how much capital banks have to hold to protect against losses."""

inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    encoder_outputs = model.encoder(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
    )

    decoder_input_ids = torch.tensor([[config.decoder_start_token_id]])

    outputs = model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        decoder_input_ids=decoder_input_ids,
    )

reference_summary = (
    "Federal regulators are moving to ease bank capital rules by revising the "
    "Basel III endgame framework, including removing a duplicative capital "
    "calculation and adjusting requirements tied to trading and mortgage lending."
)

print("Reference summary:")
print(reference_summary)
print()

print("input_ids.shape =", tuple(inputs["input_ids"].shape))
print("encoder_outputs.last_hidden_state.shape =", tuple(encoder_outputs.last_hidden_state.shape))
print("outputs.logits.shape =", tuple(outputs.logits.shape))

Reference summary:
Federal regulators are moving to ease bank capital rules by revising the Basel III endgame framework, including removing a duplicative capital calculation and adjusting requirements tied to trading and mortgage lending.

input_ids.shape = (1, 34)
encoder_outputs.last_hidden_state.shape = (1, 34, 768)
outputs.logits.shape = (1, 1, 32128)
